In [ ]:
# parameters
# BINDER_FAST: set N=6, T_ns=200, n_steps=30, n_iter_grape=50, n_iter_crab=100, n_iter_krotov=30 for fast cloud execution
N = 10               # Hilbert space truncation (Fock dimension)
omega0_GHz = 5.0     # bare cavity frequency (GHz)
K_GHz = 0.2          # Kerr nonlinearity (GHz)
T_ns = 500           # gate time (nanoseconds)
n_steps = 80         # time slices (GRAPE and Krotov)
n_iter_grape = 150   # GRAPE iteration budget
n_iter_crab = 300    # CRAB function evaluation budget
n_iter_krotov = 80   # Krotov iteration budget
n_basis_crab = 5     # CRAB: number of random Fourier basis functions
TWO_PI = 2 * 3.141592653589793

In [ ]:
# hide
import time
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.gates import snap_unitary_ideal, optimize_grape, optimize_crab, GRAPEResult
from bosonic_gates.gates.grape import optimize_krotov
from bosonic_gates.driven_kerr import DrivenKerrConfig, make_H0, make_operators

TWO_PI = 2 * np.pi

In [ ]:
# hide
from scipy.linalg import expm

def gate_fidelity(U_achieved: qt.Qobj, U_target: qt.Qobj) -> float:
    """Process fidelity F = |Tr(U†V)|² / d²."""
    d = U_target.shape[0]
    overlap = (U_achieved.dag() * U_target).tr()
    return float(np.abs(overlap)**2 / d**2)

## Module 5c: Comparison — GRAPE vs Krotov vs CRAB

**Learning objectives:**
- Understand the algorithmic differences between GRAPE, CRAB, and Krotov's method
- Run all three optimizers on the same target SNAP gate
- Compare convergence speed, final fidelity, and pulse shape
- Understand the practical trade-offs: when to use which method

---

**Sections:**
[1 Three Approaches](#sec1) ·
[2 Shared Setup](#sec2) ·
[3 Convergence Comparison](#sec3) ·
[4 Pulse Shape Comparison](#sec4) ·
[5 Trade-offs](#sec5) ·
[Exercises](#exercises)

<a id="sec1"></a>
## 1  Three Approaches to Quantum Optimal Control

Quantum optimal control (QOC) is the problem of finding a pulse $u(t)$ that
implements a target gate with maximum fidelity.  Three algorithms are in
widespread use:

---

### GRAPE — GRadient Ascent Pulse Engineering

**Reference:** Khaneja *et al.*, *J. Magn. Reson.* **172**, 296 (2005)

GRAPE represents the pulse as piecewise-constant amplitudes
$\{u_k\}_{k=1}^{n_{\rm steps}}$ and computes the fidelity gradient
$\partial F/\partial u_k$ via a forward–backward propagation sweep.
The `bosonic_gates` implementation uses JAX automatic differentiation through
the Cayley-approximant product formula, with the Adam optimizer for gradient ascent.

**Strengths:** Fast convergence for long, high-fidelity gates; scales well to
large $N$; benefits from GPU acceleration via JAX.

**Weaknesses:** No convergence guarantee (may trap in local maxima); gradient
computation requires storing all forward propagators ($\mathcal{O}(N^2 n_{\rm steps})$
memory).

---

### CRAB — Chopped Random Basis

**Reference:** Caneva *et al.*, *PRL* **106**, 190501 (2011)

CRAB parametrizes the pulse as a sum of random Fourier components:

$$u(t) = \sum_{k=1}^{K}\bigl[a_k\cos(\omega_k t) + b_k\sin(\omega_k t)\bigr],$$

where $\omega_k$ are randomly perturbed harmonics of $2\pi/T$.  Only $2Kn_{\rm ctrl}$
real parameters are optimized — far fewer than GRAPE's $n_{\rm steps}\times n_{\rm ctrl}$.
A gradient-free optimizer (Nelder-Mead) is used.

**Strengths:** Band-limited, smooth pulses by construction (hardware-friendly);
few parameters; works well for short gate times or constrained pulse shapes.

**Weaknesses:** Plateau fidelity lower than GRAPE for high-dimensional problems;
Nelder-Mead convergence is slow in many dimensions; no gradient information.

---

### Krotov's Method

**Reference:** Sklarz & Tannor, *PRA* **66**, 053619 (2002)

Krotov's method is an iterative QOC algorithm derived from variational principles.
The update rule for the control at time $t_k$ uses the current co-state
$|\chi(t_k)\rangle$ and forward state $|\psi(t_k)\rangle$:

$$u_k^{\rm new} = u_k^{\rm old} + \frac{1}{\lambda_a}\,\mathrm{Im}\!
\left[\langle\chi(t_k)|\hat{H}_{\rm ctrl}|\psi(t_k)\rangle\right],$$

where $\lambda_a > 0$ is a step-size parameter.  The key property is
**monotonic convergence**: the fidelity is guaranteed to not decrease at
each iteration, provided $\lambda_a$ is chosen large enough.

**Strengths:** Guaranteed non-decreasing fidelity at every step; natural
for open systems; good as a polishing step or verification tool.

**Weaknesses:** Updates are sequential (not parallel across time slices),
making each iteration more expensive; convergence can be slow near the optimum;
larger $\lambda_a$ gives safer but slower updates.

---

*__Lab note.__ All three methods assume a **fixed total gate time** $T$;
increasing $T$ always improves the achievable fidelity up to the decoherence
limit.  The practical question is: for a given $T$, which algorithm reaches
the highest fidelity in the fewest wall-clock seconds?  The answer depends on
the problem size, hardware (CPU vs GPU), and available libraries.*

<a id="sec2"></a>
## 2  Shared Setup

All three methods will solve the same problem: implement a two-level SNAP gate
$\theta_2 = \pi$, $\theta_5 = \pi/2$ on a Kerr oscillator.  We use identical
Hamiltonians, gate time, and random seed so that any differences in the results
come purely from the algorithm, not the problem instance.

In [ ]:
# System
cfg = DrivenKerrConfig(N=N, omega0=TWO_PI * omega0_GHz * 1e9,
                       K=TWO_PI * K_GHz * 1e9)
H0 = make_H0(cfg)
a, adag, _ = make_operators(N)
H_ctrl = [a + adag]   # single X-quadrature channel

# Target SNAP
thetas = np.zeros(N)
thetas[2] = np.pi
thetas[5] = np.pi / 2
target_U = snap_unitary_ideal(N, thetas)

T = T_ns * 1e-9  # gate time in seconds

print("=== Shared Setup ===")
print(f"System: Kerr oscillator, N={N}")
print(f"omega0/(2pi) = {cfg.omega0/TWO_PI/1e9:.3f} GHz")
print(f"K/(2pi)      = {cfg.K/TWO_PI/1e9:.3f} GHz")
print(f"Gate time T  = {T_ns} ns")
print(f"Target: SNAP with theta[2]=pi, theta[5]=pi/2")
print()
print("Optimizer parameters:")
print(f"  GRAPE:  n_steps={n_steps}, n_iter={n_iter_grape}")
print(f"  CRAB:   n_basis={n_basis_crab}, n_iter={n_iter_crab}")
print(f"  Krotov: n_steps={n_steps}, n_iter={n_iter_krotov}")

In [ ]:
# Run all three optimizers and measure wall time
print("Running GRAPE...")
t0 = time.perf_counter()
result_grape = optimize_grape(
    target_U, H0, H_ctrl, T=T,
    n_steps=n_steps, n_iter=n_iter_grape, seed=42, use_jax=True,
)
t_grape = time.perf_counter() - t0
print(f"  GRAPE done.  F = {result_grape.fidelity:.6f}  ({t_grape:.1f} s)")

print("Running CRAB...")
t0 = time.perf_counter()
result_crab = optimize_crab(
    target_U, H0, H_ctrl, T=T,
    n_basis=n_basis_crab, n_iter=n_iter_crab, seed=42,
)
t_crab = time.perf_counter() - t0
print(f"  CRAB done.   F = {result_crab.fidelity:.6f}  ({t_crab:.1f} s)")

print("Running Krotov...")
t0 = time.perf_counter()
result_krotov = optimize_krotov(
    target_U, H0, H_ctrl, T=T,
    n_steps=n_steps, n_iter=n_iter_krotov, seed=42,
)
t_krotov = time.perf_counter() - t0
print(f"  Krotov done. F = {result_krotov.fidelity:.6f}  ({t_krotov:.1f} s)")

print()
print("=" * 55)
print(f"{'Method':<12}  {'Fidelity':>10}  {'1-F':>10}  {'Wall time':>10}")
print("-" * 55)
for label, res, wall_t in [
    ('GRAPE',  result_grape,  t_grape),
    ('CRAB',   result_crab,   t_crab),
    ('Krotov', result_krotov, t_krotov),
]:
    print(f"{label:<12}  {res.fidelity:>10.6f}  {res.infidelity:>10.2e}  {wall_t:>8.1f} s")

<a id="sec3"></a>
## 3  Convergence Comparison

We plot the infidelity $1-F$ as a function of iteration number for all three
methods on the same axes.  Note: the x-axis counts *iterations*, not wall time.
A single Krotov iteration (full forward + backward sweep) is more expensive
than a single GRAPE gradient step, and each CRAB function evaluation is cheaper
still — so the iteration count does not directly measure computational cost.

The wall times printed above give the true comparison.

In [ ]:
# Infidelity history for all three methods
fig, ax = plt.subplots(figsize=(9, 5))

methods = [
    (result_grape,  'GRAPE',  'steelblue',   '-',  t_grape),
    (result_crab,   'CRAB',   'darkorange',  '--', t_crab),
    (result_krotov, 'Krotov', 'forestgreen', '-.', t_krotov),
]

for res, label, color, ls, wall_t in methods:
    infids = [1.0 - f for f in res.fidelity_history]
    iters  = np.arange(1, len(infids) + 1)
    ax.semilogy(iters, infids, color=color, ls=ls, lw=2,
                label=(
                    rf"{label}: $F_{{\rm final}}={res.fidelity:.4f}$, "
                    rf"$1-F={res.infidelity:.2e}$, {wall_t:.1f} s"
                ))

ax.set_xlabel('Iteration')
ax.set_ylabel(r'Infidelity $1-F$')
ax.set_title(
    rf'Convergence comparison: GRAPE vs CRAB vs Krotov ($N={N}$, $T={T_ns}$ ns)',
    fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(1, None)
plt.tight_layout()
plt.show()

# Print summary
print("Final fidelities:")
for res, label, *_ in methods:
    print(f"  {label:<8} F = {res.fidelity:.6f}  (1-F = {res.infidelity:.2e})")

**Observation.** The three methods have qualitatively different convergence profiles:

- **GRAPE** shows rapid early progress (gradient points directly uphill) followed
  by an asymptotic plateau as it approaches a local optimum.  The Adam optimizer's
  adaptive learning rate sustains progress longer than vanilla gradient ascent.

- **CRAB** starts from a random Fourier parametrization and explores a much
  lower-dimensional space ($2Kn_{\rm ctrl}$ parameters vs $n_{\rm steps}n_{\rm ctrl}$
  for GRAPE).  Nelder-Mead convergence is slower per iteration but each evaluation
  is cheaper.  The final fidelity plateau is typically lower than GRAPE for
  the same gate time.

- **Krotov** shows *monotone* convergence — every iteration increases $F$ or
  leaves it unchanged.  Progress is steady but conservative, particularly
  near the optimum where the co-state correction becomes small.

---
*__Lab note.__ Wall-time comparison is hardware-dependent.  On a CPU, GRAPE
with JAX is typically fastest because JAX JIT-compiles the gradient computation
to efficient BLAS operations.  On a GPU, GRAPE can be orders of magnitude faster
than Krotov, which requires sequential slice updates.  CRAB is always the fastest
per function evaluation but may need many more evaluations to reach the same fidelity.*

<a id="sec4"></a>
## 4  Pulse Shape Comparison

The three methods produce qualitatively different pulse shapes:

- **GRAPE and Krotov** both use piecewise-constant pulses — they can represent any
  arbitrary waveform at the given time resolution.
- **CRAB** constructs the pulse as a sum of sinusoids — the result is a smooth,
  band-limited waveform with at most $n_{\rm basis}$ harmonics.

The pulse shape has direct experimental relevance: AWGs have finite bandwidth,
and CRAB's smooth pulses are reproduced more faithfully than GRAPE's
piecewise-constant pulses (which have high-frequency Gibbs ringing after
AWG filtering).

In [ ]:
tlist_ns = np.linspace(0, T_ns, n_steps)
tlist_ns_crab = np.linspace(0, T_ns, len(result_crab.pulse_params[0]))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle(
    rf"Pulse shapes and convergence: $N={N}$, $T={T_ns}$ ns",
    fontsize=12)

# Row 0: pulse shapes
# GRAPE
axes[0, 0].step(tlist_ns, result_grape.pulse_params[0], where='post',
                lw=1.5, color='steelblue')
axes[0, 0].set_title(rf"GRAPE pulse ($n_{{\rm steps}}={n_steps}$)")
axes[0, 0].set_xlabel('Time (ns)')
axes[0, 0].set_ylabel('Amplitude $u_k$ (rad/s)')
axes[0, 0].axhline(0, color='gray', lw=0.8, ls='--')

# CRAB
axes[0, 1].plot(tlist_ns_crab, result_crab.pulse_params[0],
                lw=1.5, color='darkorange')
axes[0, 1].set_title(rf"CRAB pulse ($n_{{\rm basis}}={n_basis_crab}$, smooth)")
axes[0, 1].set_xlabel('Time (ns)')
axes[0, 1].set_ylabel('Amplitude $u(t)$ (rad/s)')
axes[0, 1].axhline(0, color='gray', lw=0.8, ls='--')

# Krotov
axes[0, 2].step(tlist_ns, result_krotov.pulse_params[0], where='post',
                lw=1.5, color='forestgreen')
axes[0, 2].set_title(rf"Krotov pulse ($n_{{\rm steps}}={n_steps}$)")
axes[0, 2].set_xlabel('Time (ns)')
axes[0, 2].set_ylabel('Amplitude $u_k$ (rad/s)')
axes[0, 2].axhline(0, color='gray', lw=0.8, ls='--')

# Row 1: convergence curves
for ax, res, label, color in [
    (axes[1, 0], result_grape,  'GRAPE',  'steelblue'),
    (axes[1, 1], result_crab,   'CRAB',   'darkorange'),
    (axes[1, 2], result_krotov, 'Krotov', 'forestgreen'),
]:
    infids = [1.0 - f for f in res.fidelity_history]
    ax.semilogy(np.arange(1, len(infids)+1), infids, lw=2, color=color)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(r'Infidelity $1-F$')
    ax.set_title(
        rf"{label}: $F = {res.fidelity:.4f}$")

plt.tight_layout()
plt.show()

In [ ]:
# Frequency-domain comparison of pulse shapes
fig, ax = plt.subplots(figsize=(9, 4))

for res, label, color in [
    (result_grape,  'GRAPE',  'steelblue'),
    (result_crab,   'CRAB',   'darkorange'),
    (result_krotov, 'Krotov', 'forestgreen'),
]:
    pulse = res.pulse_params[0]
    n = len(pulse)
    freq_GHz = np.fft.rfftfreq(n, d=T_ns/n) * 1e-0  # MHz
    spectrum = np.abs(np.fft.rfft(pulse))
    ax.semilogy(freq_GHz, spectrum / spectrum.max(), lw=1.5, label=label, color=color)

ax.set_xlabel(rf'Frequency (1/ns = GHz, at {T_ns} ns total)')
ax.set_ylabel('Normalized spectral amplitude')
ax.set_title('Pulse spectrum: GRAPE vs CRAB vs Krotov')
ax.legend()
ax.set_xlim(0, None)
plt.tight_layout()
plt.show()

# Bandwidth (RMS frequency)
print("Pulse RMS amplitudes:")
for res, label in [(result_grape, 'GRAPE'), (result_crab, 'CRAB'), (result_krotov, 'Krotov')]:
    rms = float(np.sqrt(np.mean(res.pulse_params[0]**2)))
    print(f"  {label:<8}  RMS = {rms:.4f} rad/s")

**Observation.** The CRAB pulse spectrum drops off sharply at $n_{\rm basis}/T$
because it contains only $n_{\rm basis}$ frequency components.  GRAPE and Krotov
pulses have broad spectra due to the piecewise-constant nature — all frequencies
up to $1/\delta t$ are present (though the amplitude rolls off).  AWG filtering
will affect GRAPE/Krotov pulses more than CRAB pulses.

---
*__Lab note.__ In experiment, the pulse is typically band-limited to $\sim 500$ MHz
(for a 1 GS/s AWG) or $\sim 2$ GHz (for direct synthesis at microwave frequencies).
CRAB with a suitable $n_{\rm basis}$ automatically respects this bandwidth constraint.
For GRAPE, a post-processing filter or a bandwidth penalty term in the objective
must be added explicitly.*

<a id="sec5"></a>
## 5  Trade-offs and Method Selection

The three methods have complementary strengths.  The table below summarizes
the key practical trade-offs:

| Method | Gradient used | Convergence guarantee | Free parameters | Best use case |
|---|---|---|---|---|
| **GRAPE** | Yes (autodiff) | Not guaranteed (local max) | $n_{\rm steps} \times n_{\rm ctrl}$ | Long, high-fidelity gates; GPU available |
| **CRAB** | No (Nelder-Mead) | Not guaranteed | $2 \times n_{\rm basis} \times n_{\rm ctrl}$ | Short gates; bandwidth-constrained; hardware-friendly pulses |
| **Krotov** | Implicit (co-state) | Monotone (non-decreasing $F$) | $n_{\rm steps} \times n_{\rm ctrl}$ | Reliable polishing; verification; open systems |

**Practical recommendations:**

1. **Start with GRAPE** (JAX backend) for most cavity gate optimization tasks.
   Use multiple random seeds (`seed=0,1,2,...`) to escape local maxima.

2. **Use CRAB** when the pulse must be band-limited, when JAX is not available,
   or for short gate times ($T K \lesssim 2\pi$) where fewer parameters suffice.

3. **Use Krotov** as a verification tool or polishing step after GRAPE:
   start from the GRAPE optimum and run a few Krotov iterations.
   The monotone convergence guarantees you cannot do worse.

4. **For open-system gates** (with collapse operators), all three methods can
   accept `c_ops`.  Krotov's derivation naturally handles Lindblad dynamics
   and is often preferred for strongly dissipative systems.

In [ ]:
# Summary table with measured results
print(f"{'='*65}")
print(f"{'Method':<10} {'F':>8} {'1-F':>10} {'Iters':>8} {'Wall-time':>11}")
print(f"{'-'*65}")
for (res, label, *_), wall_t in [
    ((result_grape,  'GRAPE',  'steelblue',   '-',  t_grape),  t_grape),
    ((result_crab,   'CRAB',   'darkorange',  '--', t_crab),   t_crab),
    ((result_krotov, 'Krotov', 'forestgreen', '-.', t_krotov), t_krotov),
]:
    n_it = len(res.fidelity_history)
    print(f"{label:<10} {res.fidelity:>8.5f} {res.infidelity:>10.2e} {n_it:>8d} {wall_t:>9.1f} s")
print(f"{'='*65}")

# Which method won?
best_label, best_F = max(
    [('GRAPE', result_grape.fidelity),
     ('CRAB',  result_crab.fidelity),
     ('Krotov', result_krotov.fidelity)],
    key=lambda x: x[1]
)
print(f"\nBest fidelity: {best_label} with F = {best_F:.6f}")

In [ ]:
# Combined convergence on one plot with normalized x-axis (fraction of budget)
fig, ax = plt.subplots(figsize=(8, 5))

for res, label, color, ls, _ in [
    (result_grape,  'GRAPE',  'steelblue',   '-',  t_grape),
    (result_crab,   'CRAB',   'darkorange',  '--', t_crab),
    (result_krotov, 'Krotov', 'forestgreen', '-.', t_krotov),
]:
    infids = [1.0 - f for f in res.fidelity_history]
    fraction = np.linspace(0, 1, len(infids))
    ax.semilogy(fraction, infids, color=color, ls=ls, lw=2,
                label=rf"{label}: $F_{{\rm final}}={res.fidelity:.4f}$")

ax.set_xlabel('Fraction of iteration budget')
ax.set_ylabel(r'Infidelity $1-F$')
ax.set_title(
    rf"Convergence vs iteration budget fraction ($N={N}$, $T={T_ns}$ ns)",
    fontsize=11)
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

---
*__Lab note.__ In practice, GRAPE with JAX autodiff is the fastest method for
cavity gate optimization on modern hardware.  Krotov is preferred when you need
a reliability guarantee — the monotone convergence property means a Krotov run
will never report a lower fidelity than its starting point, making it ideal for
verifying solutions found by GRAPE.  CRAB is used when pulse bandwidth must be
limited (e.g., room-temperature electronics with 100 MHz AWG bandwidth) or when
the optimization must run on a device without JAX support.*

<a id="exercises"></a>
## Exercises

**Exercise 1.** Halve `n_steps` (coarser time grid) and re-run all three methods.
Does each method degrade similarly, or does one show more sensitivity to
the time resolution?  Plot the new convergence curves alongside the original.

**Exercise 2.** Run all three methods with `seed=0` instead of `seed=42`.
Do they converge to the same final fidelity?  Same pulse shape?  What does this
tell you about the fidelity landscape?

In [ ]:
# Exercise 1 — coarser time grid (n_steps // 2)
# YOUR CODE HERE

In [ ]:
# Exercise 2 — same seed=0 for all three methods
# YOUR CODE HERE

In [ ]:
# Solution — Exercise 1: coarser time grid
n_steps_coarse = n_steps // 2
print(f"Coarse grid: n_steps = {n_steps_coarse} (was {n_steps})")

res_grape_c = optimize_grape(target_U, H0, H_ctrl, T=T,
                             n_steps=n_steps_coarse, n_iter=n_iter_grape, seed=42)
res_crab_c  = optimize_crab(target_U, H0, H_ctrl, T=T,
                            n_basis=n_basis_crab, n_iter=n_iter_crab, seed=42)
res_krotov_c = optimize_krotov(target_U, H0, H_ctrl, T=T,
                               n_steps=n_steps_coarse, n_iter=n_iter_krotov, seed=42)

print(f"{'Method':<10}  {'F (fine)':>12}  {'F (coarse)':>12}  {'degradation':>12}")
for res_f, res_c, label in [
    (result_grape,  res_grape_c,  'GRAPE'),
    (result_crab,   res_crab_c,   'CRAB'),
    (result_krotov, res_krotov_c, 'Krotov'),
]:
    print(f"{label:<10}  {res_f.fidelity:>12.6f}  {res_c.fidelity:>12.6f}"
          f"  {res_f.fidelity - res_c.fidelity:>12.6f}")

# Convergence comparison
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (res_f, res_c, label, color) in zip(axes, [
    (result_grape,  res_grape_c,  'GRAPE',  'steelblue'),
    (result_crab,   res_crab_c,   'CRAB',   'darkorange'),
    (result_krotov, res_krotov_c, 'Krotov', 'forestgreen'),
]):
    ax.semilogy([1-f for f in res_f.fidelity_history], color=color, lw=2, label=f'n_steps={n_steps}')
    ax.semilogy([1-f for f in res_c.fidelity_history], color=color, lw=2, ls='--', label=f'n_steps={n_steps_coarse}')
    ax.set_xlabel('Iteration')
    ax.set_ylabel(r'$1-F$')
    ax.set_title(label)
    ax.legend(fontsize=8)
plt.suptitle('Effect of coarser time grid on convergence', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Solution — Exercise 2: different seed
res_grape_s0  = optimize_grape(target_U, H0, H_ctrl, T=T, n_steps=n_steps,
                               n_iter=n_iter_grape, seed=0)
res_crab_s0   = optimize_crab(target_U, H0, H_ctrl, T=T, n_basis=n_basis_crab,
                              n_iter=n_iter_crab, seed=0)
res_krotov_s0 = optimize_krotov(target_U, H0, H_ctrl, T=T, n_steps=n_steps,
                                n_iter=n_iter_krotov, seed=0)

print("Final fidelities — seed 42 vs seed 0:")
print(f"{'Method':<10}  {'seed=42':>10}  {'seed=0':>10}  {'difference':>12}")
for res42, res0, label in [
    (result_grape,  res_grape_s0,  'GRAPE'),
    (result_crab,   res_crab_s0,   'CRAB'),
    (result_krotov, res_krotov_s0, 'Krotov'),
]:
    print(f"{label:<10}  {res42.fidelity:>10.6f}  {res0.fidelity:>10.6f}"
          f"  {abs(res42.fidelity-res0.fidelity):>12.6f}")

print("\nConclusion: different seeds → different local optima → different final F.")
print("This shows the fidelity landscape has multiple local maxima.")